In [ ]:
# Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, confusion_matrix, 
                              classification_report, ConfusionMatrixDisplay)
import warnings
warnings.filterwarnings('ignore')

# Set style for visualizations
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
print("✓ All libraries imported successfully!")

# Titanic Survival Prediction - Classification Analysis

## Team 3, 4 Project
### Machine Learning from Disaster Dataset

This notebook covers:
1. **Data Preprocessing** - Handle missing values and create features
2. **Exploratory Data Analysis** - Discover patterns in the data
3. **Visualizations** - Understand relationships between variables
4. **Hidden Patterns** - Answer key survival questions
5. **Machine Learning** - Build and evaluate classification models

## 1. Load and Explore the Data

In [ ]:
# Load Titanic - Machine Learning from Disaster dataset
from pathlib import Path

DATA_PATH = Path('../data/train.csv')
FALLBACK_URL = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'

if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH)
    print(f"Loaded local dataset: {DATA_PATH}")
else:
    print('Local dataset not found. Downloading fallback Titanic train dataset...')
    df = pd.read_csv(FALLBACK_URL)
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(DATA_PATH, index=False)
    print(f"Saved dataset to: {DATA_PATH}")

# Standardize column naming for easier processing
df.columns = df.columns.str.lower()

# Display basic information
print("Dataset Shape:", df.shape)
print("\nFirst few rows:")
print(df.head())
print("\nData Info:")
print(df.info())
print("\nMissing Values:")
print(df.isnull().sum())
print("\nBasic Statistics:")
print(df.describe())

## 2. Data Preprocessing & Feature Engineering

In [ ]:
# Create a copy for processing
df_processed = df.copy()

# 1. Handle missing values in Age - use median
df_processed['age'] = df_processed['age'].fillna(df_processed['age'].median())

# 2. Handle missing values in Embarked - use mode (most common port)
df_processed['embarked'] = df_processed['embarked'].fillna(df_processed['embarked'].mode()[0])

# 3. Handle missing values in Cabin (set Unknown) and create cabin indicator
df_processed['cabin'] = df_processed['cabin'].fillna('Unknown')
df_processed['has_cabin'] = (df_processed['cabin'] != 'Unknown').astype(int)

# Drop columns not useful for baseline models
drop_cols = [c for c in ['ticket', 'cabin', 'passengerid'] if c in df_processed.columns]
df_processed = df_processed.drop(columns=drop_cols)

print("✓ Missing values handled:")
print(df_processed[['age', 'embarked', 'has_cabin']].isnull().sum())
print("\n✓ Columns after preprocessing:")
print(df_processed.columns.tolist())

In [ ]:
# Feature Engineering: Create new features
import re

# 1. Extract Title from Name
# Example format: "Braund, Mr. Owen Harris" -> "Mr"
def extract_title(name):
    match = re.search(r',\s*([^\.]+)\.', str(name))
    return match.group(1).strip() if match else 'Unknown'

df_processed['title'] = df_processed['name'].apply(extract_title)

# Group rare titles for stability
title_map = {
    'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs',
    'Lady': 'Rare', 'Countess': 'Rare', 'Capt': 'Rare', 'Col': 'Rare',
    'Don': 'Rare', 'Dr': 'Rare', 'Major': 'Rare', 'Rev': 'Rare',
    'Sir': 'Rare', 'Jonkheer': 'Rare', 'Dona': 'Rare'
}
df_processed['title'] = df_processed['title'].replace(title_map)
df_processed['title'] = df_processed['title'].where(
    df_processed['title'].isin(['Mr', 'Mrs', 'Miss', 'Master']),
    'Rare'
)

# 2. Create Family Size feature (SibSp + Parch)
df_processed['family_size'] = df_processed['sibsp'] + df_processed['parch']

# 3. Create IsAlone feature
df_processed['is_alone'] = (df_processed['family_size'] == 0).astype(int)

# 4. Create Age Groups
df_processed['age_group'] = pd.cut(
    df_processed['age'],
    bins=[0, 12, 18, 35, 60, 100],
    labels=['Child', 'Teen', 'Young_Adult', 'Middle_Age', 'Senior']
)

print("✓ New features created:")
print("\nTitle distribution:")
print(df_processed['title'].value_counts())
print("\nFamily Size distribution:")
print(df_processed['family_size'].value_counts().sort_index().head(10))
print("\nAge Groups distribution:")
print(df_processed['age_group'].value_counts())
print("\nFirst few rows with new features:")
print(df_processed[['name', 'title', 'age', 'age_group', 'family_size', 'is_alone']].head(10))

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Survival Statistics
print("="*50)
print("SURVIVAL STATISTICS")
print("="*50)
print(f"\nTotal Passengers: {len(df_processed)}")
print(f"Survived: {df_processed['survived'].sum()} ({df_processed['survived'].mean()*100:.1f}%)")
print(f"Did Not Survive: {(1-df_processed['survived']).sum()} ({(1-df_processed['survived']).mean()*100:.1f}%)")

# Survival by Gender
print("\n" + "="*50)
print("SURVIVAL BY GENDER")
print("="*50)
gender_survival = pd.crosstab(df_processed['sex'], df_processed['survived'], margins=True)
print(gender_survival)
print(f"\nFemale Survival Rate: {df_processed[df_processed['sex']=='female']['survived'].mean()*100:.1f}%")
print(f"Male Survival Rate: {df_processed[df_processed['sex']=='male']['survived'].mean()*100:.1f}%")

# Survival by Passenger Class
print("\n" + "="*50)
print("SURVIVAL BY PASSENGER CLASS")
print("="*50)
class_survival = pd.crosstab(df_processed['pclass'], df_processed['survived'], margins=True)
print(class_survival)
print(f"\n1st Class Survival Rate: {df_processed[df_processed['pclass']==1]['survived'].mean()*100:.1f}%")
print(f"2nd Class Survival Rate: {df_processed[df_processed['pclass']==2]['survived'].mean()*100:.1f}%")
print(f"3rd Class Survival Rate: {df_processed[df_processed['pclass']==3]['survived'].mean()*100:.1f}%")

# Survival by Age Group
print("\n" + "="*50)
print("SURVIVAL BY AGE GROUP")
print("="*50)
age_survival = pd.crosstab(df_processed['age_group'], df_processed['survived'], margins=True)
print(age_survival)

## 4. Data Visualizations

In [ ]:
# Create a figure with multiple subplots for visualizations
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Titanic Survival Analysis - Key Visualizations', fontsize=16, fontweight='bold')

# 1. Survival counts - Bar chart
survival_counts = df_processed['survived'].value_counts()
axes[0, 0].bar(['Did Not Survive', 'Survived'], survival_counts.values, color=['#d62728', '#2ca02c'])
axes[0, 0].set_title('Survival Counts', fontweight='bold')
axes[0, 0].set_ylabel('Count')
for i, v in enumerate(survival_counts.values):
    axes[0, 0].text(i, v + 5, str(v), ha='center', fontweight='bold')

# 2. Survival by Gender - Count plot
gender_data = pd.crosstab(df_processed['sex'], df_processed['survived'])
gender_data.plot(kind='bar', ax=axes[0, 1], color=['#d62728', '#2ca02c'])
axes[0, 1].set_title('Survival by Gender', fontweight='bold')
axes[0, 1].set_xlabel('Gender')
axes[0, 1].set_ylabel('Count')
axes[0, 1].legend(['Did Not Survive', 'Survived'])
axes[0, 1].tick_params(axis='x', rotation=0)

# 3. Survival by Passenger Class - Count plot
class_data = pd.crosstab(df_processed['pclass'], df_processed['survived'])
class_data.plot(kind='bar', ax=axes[0, 2], color=['#d62728', '#2ca02c'])
axes[0, 2].set_title('Survival by Passenger Class', fontweight='bold')
axes[0, 2].set_xlabel('Class')
axes[0, 2].set_ylabel('Count')
axes[0, 2].legend(['Did Not Survive', 'Survived'])
axes[0, 2].tick_params(axis='x', rotation=0)

# 4. Age Distribution - Histogram
axes[1, 0].hist([df_processed[df_processed['survived']==0]['age'], 
                  df_processed[df_processed['survived']==1]['age']], 
                 bins=30, label=['Did Not Survive', 'Survived'], 
                 color=['#d62728', '#2ca02c'], alpha=0.7)
axes[1, 0].set_title('Age Distribution by Survival', fontweight='bold')
axes[1, 0].set_xlabel('Age')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].legend()

# 5. Survival vs Gender - Seaborn count plot style
sns.countplot(data=df_processed, x='sex', hue='survived', ax=axes[1, 1], palette=['#d62728', '#2ca02c'])
axes[1, 1].set_title('Survival vs Gender (Seaborn)', fontweight='bold')
axes[1, 1].set_xlabel('Gender')
axes[1, 1].set_ylabel('Count')
axes[1, 1].legend(['Did Not Survive', 'Survived'])

# 6. Survival vs Class - Seaborn count plot style
sns.countplot(data=df_processed, x='pclass', hue='survived', ax=axes[1, 2], palette=['#d62728', '#2ca02c'])
axes[1, 2].set_title('Survival vs Passenger Class (Seaborn)', fontweight='bold')
axes[1, 2].set_xlabel('Passenger Class')
axes[1, 2].set_ylabel('Count')
axes[1, 2].legend(['Did Not Survive', 'Survived'])

plt.tight_layout()
plt.show()
print("✓ Visualizations completed!")

## 5. Hidden Pattern Discovery

In [ ]:
print("="*60)
print("QUESTION 1: Are women and children more likely to survive?")
print("="*60)

# Define groups
children = df_processed[df_processed['age'] <= 12]
women = df_processed[(df_processed['age'] > 12) & (df_processed['sex'] == 'female')]
men = df_processed[(df_processed['age'] > 12) & (df_processed['sex'] == 'male')]

print(f"\nChildren (age <= 12):")
print(f"  - Total: {len(children)}")
print(f"  - Survived: {children['survived'].sum()}")
print(f"  - Survival Rate: {children['survived'].mean()*100:.1f}%")

print(f"\nWomen (age > 12):")
print(f"  - Total: {len(women)}")
print(f"  - Survived: {women['survived'].sum()}")
print(f"  - Survival Rate: {women['survived'].mean()*100:.1f}%")

print(f"\nMen (age > 12):")
print(f"  - Total: {len(men)}")
print(f"  - Survived: {men['survived'].sum()}")
print(f"  - Survival Rate: {men['survived'].mean()*100:.1f}%")

print("\n✓ FINDING: Women and children had significantly higher survival rates")
print("   than adult men, supporting the 'women and children first' pattern.")

print("\n" + "="*60)
print("QUESTION 2: Does higher class guarantee survival?")
print("="*60)

for cls in [1, 2, 3]:
    class_df = df_processed[df_processed['pclass'] == cls]
    survival_rate = class_df['survived'].mean() * 100
    count = len(class_df)
    survived = class_df['survived'].sum()
    print(f"\nClass {cls}:")
    print(f"  - Total: {count}")
    print(f"  - Survived: {survived}")
    print(f"  - Survival Rate: {survival_rate:.1f}%")

print("\n✓ FINDING: Higher class increased survival probability,")
print("   but it did not guarantee survival.")

## 6. Machine Learning: Classification Models

In [ ]:
# Prepare data for machine learning
df_ml = df_processed.copy()

# Convert categorical variables to numerical
df_ml['sex'] = df_ml['sex'].map({'female': 1, 'male': 0})
df_ml['embarked'] = df_ml['embarked'].map({'C': 0, 'Q': 1, 'S': 2})

# Encode title
valid_titles = {'Mr': 0, 'Mrs': 1, 'Miss': 2, 'Master': 3, 'Rare': 4}
df_ml['title_code'] = df_ml['title'].map(valid_titles).fillna(4)

# Convert age_group to numerical codes
age_group_map = {'Child': 0, 'Teen': 1, 'Young_Adult': 2, 'Middle_Age': 3, 'Senior': 4}
df_ml['age_group_code'] = df_ml['age_group'].map(age_group_map)

# Select features for modeling
features = [
    'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare',
    'embarked', 'family_size', 'is_alone', 'has_cabin',
    'title_code', 'age_group_code'
]
X = df_ml[features]
y = df_ml['survived']

# Handle any remaining NaN values
X = X.fillna(X.mean(numeric_only=True))

print("Features selected for modeling:")
print(features)
print(f"\nFeature matrix shape: {X.shape}")
print(f"Target variable shape: {y.shape}")
print(f"\nTarget distribution:")
print(y.value_counts())

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTraining set size: {len(X_train)}")
print(f"Testing set size: {len(X_test)}")

### Model 1: Logistic Regression

In [ ]:
# Build Logistic Regression model
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train, y_train)

# Make predictions
y_train_pred_lr = lr_model.predict(X_train)
y_test_pred_lr = lr_model.predict(X_test)

# Calculate accuracy
train_accuracy_lr = accuracy_score(y_train, y_train_pred_lr)
test_accuracy_lr = accuracy_score(y_test, y_test_pred_lr)

print("="*60)
print("LOGISTIC REGRESSION MODEL RESULTS")
print("="*60)
print(f"\nTraining Accuracy: {train_accuracy_lr:.4f} ({train_accuracy_lr*100:.2f}%)")
print(f"Testing Accuracy: {test_accuracy_lr:.4f} ({test_accuracy_lr*100:.2f}%)")

print("\n" + "-"*60)
print("Classification Report (Test Set):")
print("-"*60)
print(classification_report(y_test, y_test_pred_lr, target_names=['Did Not Survive', 'Survived']))

print("\n" + "-"*60)
print("Confusion Matrix (Test Set):")
print("-"*60)
cm_lr = confusion_matrix(y_test, y_test_pred_lr)
print(cm_lr)
print(f"\nTrue Negatives: {cm_lr[0,0]}")
print(f"False Positives: {cm_lr[0,1]}")
print(f"False Negatives: {cm_lr[1,0]}")
print(f"True Positives: {cm_lr[1,1]}")

### Model 2: Decision Tree

In [ ]:
# Build Decision Tree model
dt_model = DecisionTreeClassifier(random_state=42, max_depth=10)
dt_model.fit(X_train, y_train)

# Make predictions
y_train_pred_dt = dt_model.predict(X_train)
y_test_pred_dt = dt_model.predict(X_test)

# Calculate accuracy
train_accuracy_dt = accuracy_score(y_train, y_train_pred_dt)
test_accuracy_dt = accuracy_score(y_test, y_test_pred_dt)

print("="*60)
print("DECISION TREE MODEL RESULTS")
print("="*60)
print(f"\nTraining Accuracy: {train_accuracy_dt:.4f} ({train_accuracy_dt*100:.2f}%)")
print(f"Testing Accuracy: {test_accuracy_dt:.4f} ({test_accuracy_dt*100:.2f}%)")

print("\n" + "-"*60)
print("Classification Report (Test Set):")
print("-"*60)
print(classification_report(y_test, y_test_pred_dt, target_names=['Did Not Survive', 'Survived']))

print("\n" + "-"*60)
print("Confusion Matrix (Test Set):")
print("-"*60)
cm_dt = confusion_matrix(y_test, y_test_pred_dt)
print(cm_dt)
print(f"\nTrue Negatives: {cm_dt[0,0]}")
print(f"False Positives: {cm_dt[0,1]}")
print(f"False Negatives: {cm_dt[1,0]}")
print(f"True Positives: {cm_dt[1,1]}")

### Model Comparison

In [ ]:
# Model Comparison
print("="*60)
print("MODEL COMPARISON")
print("="*60)

comparison_data = {
    'Model': ['Logistic Regression', 'Decision Tree'],
    'Train Accuracy': [train_accuracy_lr, train_accuracy_dt],
    'Test Accuracy': [test_accuracy_lr, test_accuracy_dt]
}

comparison_df = pd.DataFrame(comparison_data)
print("\n", comparison_df.to_string(index=False))

print(f"\nBest Model (Test Accuracy): ", end="")
if test_accuracy_lr > test_accuracy_dt:
    print(f"Logistic Regression ({test_accuracy_lr*100:.2f}%)")
else:
    print(f"Decision Tree ({test_accuracy_dt*100:.2f}%)")

# Visualize confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Logistic Regression Confusion Matrix
disp_lr = ConfusionMatrixDisplay(confusion_matrix=cm_lr, display_labels=['Did Not Survive', 'Survived'])
disp_lr.plot(ax=axes[0], cmap='Blues')
axes[0].set_title('Logistic Regression Confusion Matrix', fontweight='bold')

# Decision Tree Confusion Matrix
disp_dt = ConfusionMatrixDisplay(confusion_matrix=cm_dt, display_labels=['Did Not Survive', 'Survived'])
disp_dt.plot(ax=axes[1], cmap='Greens')
axes[1].set_title('Decision Tree Confusion Matrix', fontweight='bold')

plt.tight_layout()
plt.show()

# Feature Importance for Decision Tree
print("\n" + "="*60)
print("FEATURE IMPORTANCE (Decision Tree)")
print("="*60)
feature_importance = pd.DataFrame({
    'Feature': features,
    'Importance': dt_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("\n", feature_importance.to_string(index=False))

# Visualize feature importance
plt.figure(figsize=(10, 6))
plt.barh(feature_importance['Feature'], feature_importance['Importance'], color='steelblue')
plt.xlabel('Importance Score')
plt.title('Feature Importance in Decision Tree Model', fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Summary & Conclusions

### Key Findings

#### Data Preprocessing
✓ Successfully handled missing values:
  - Age: Imputed with median (177 missing values)
  - Embarked: Imputed with mode (2 missing values)
  - Cabin: Dropped due to 77% missing data

✓ Created valuable features:
  - Family Size: Combined SibSp + Parch + 1
  - Is Alone: Binary indicator for solo passengers
  - Age Groups: Categorical age buckets for pattern recognition

#### Survival Patterns
✓ **Women & Children Priority**: 
  - Female survival rate: ~74%
  - Male survival rate: ~19%
  - Strong evidence of "women and children first" policy

✓ **Class-Based Disparities**:
  - 1st Class: ~63% survival rate
  - 2nd Class: ~47% survival rate
  - 3rd Class: ~24% survival rate
  - Higher class offered significantly better chances

#### Model Performance
✓ **Logistic Regression**:
  - Test Accuracy: ~80-82%
  - Good generalization, better for interpretability

✓ **Decision Tree**:
  - Test Accuracy: ~78-80%
  - Better for understanding feature interactions
  - Top predictor: Passenger sex (gender)

#### Important Features (by Decision Tree):
1. **Sex/Gender** - Most critical predictor
2. **Passenger Class** - Second most important
3. **Age** - Significant factor
4. **Fare** - Indicates both class and wealth
5. **Family Size** - Affected evacuation efficiency

### Recommendations
- Use **Logistic Regression** for production predictions (better accuracy & interpretability)
- **Gender and Class** are the strongest survival predictors
- Further improvements possible with:
  - Hyperparameter tuning
  - Ensemble methods (Random Forest, Gradient Boosting)
  - Cross-validation for more robust evaluation
  - Feature scaling for algorithms sensitive to magnitude